In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, count, avg, desc
import pandas as pd

In [2]:
# Start a Spark session
spark = SparkSession.builder.appName("JobPostingsAnalysis").getOrCreate()

# Load the CSV file into a Spark DataFrame
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine", "true").option("escape", "\"").csv("lightcast_job_postings.csv")

# Show schema and first few rows
df.printSchema()
df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/10 22:02:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- ID: string (nullable = true)
 |-- LAST_UPDATED_DATE: string (nullable = true)
 |-- LAST_UPDATED_TIMESTAMP: timestamp (nullable = true)
 |-- DUPLICATES: integer (nullable = true)
 |-- POSTED: string (nullable = true)
 |-- EXPIRED: string (nullable = true)
 |-- DURATION: integer (nullable = true)
 |-- SOURCE_TYPES: string (nullable = true)
 |-- SOURCES: string (nullable = true)
 |-- URL: string (nullable = true)
 |-- ACTIVE_URLS: string (nullable = true)
 |-- ACTIVE_SOURCES_INFO: string (nullable = true)
 |-- TITLE_RAW: string (nullable = true)
 |-- BODY: string (nullable = true)
 |-- MODELED_EXPIRED: string (nullable = true)
 |-- MODELED_DURATION: integer (nullable = true)
 |-- COMPANY: integer (nullable = true)
 |-- COMPANY_NAME: string (nullable = true)
 |-- COMPANY_RAW: string (nullable = true)
 |-- COMPANY_IS_STAFFING: boolean (nullable = true)
 |-- EDUCATION_LEVELS: string (nullable = true)
 |-- EDUCATION_LEVELS_NAME: string (nullable = true)
 |-- MIN_EDULEVELS: integer (

25/12/10 22:02:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+-----------------+----------------------+----------+--------+---------+--------+--------------------+--------------------+--------------------+-----------+-------------------+--------------------+--------------------+---------------+----------------+--------+--------------------+-----------+-------------------+----------------+---------------------+-------------+-------------------+-------------+------------------+---------------+--------------------+--------------------+--------------------+-------------+------+-----------+----------------+-------------------+---------+-----------+--------------------+--------------------+-------------+------+--------------+-----+--------------------+-----+----------+---------------+--------------------+---------------+--------------------+------------+--------------------+------------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+------

In [3]:
# Impute missing salary values using the median salary
if "salary" in df.columns:
    salary_median = df.approxQuantile("salary", [0.5], 0.0)[0]
    df = df.na.fill({"salary": salary_median})

# Parse posting date strings into proper date type
if "date_posted" in df.columns:
    df = df.withColumn(
        "date_posted",
        to_date(col("date_posted"), "yyyy-MM-dd")
    )

# Preview a few cleaned records
df.limit(5).show()


+--------------------+-----------------+----------------------+----------+--------+---------+--------+--------------------+--------------------+--------------------+-----------+-------------------+--------------------+--------------------+---------------+----------------+--------+--------------------+-----------+-------------------+----------------+---------------------+-------------+-------------------+-------------+------------------+---------------+--------------------+--------------------+--------------------+-------------+------+-----------+----------------+-------------------+---------+-----------+--------------------+--------------------+-------------+------+--------------+-----+--------------------+-----+----------+---------------+--------------------+---------------+--------------------+------------+--------------------+------------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+------

In [4]:
from pyspark.sql.functions import count, avg, desc
import pandas as pd

# 1. Compute how many job postings are in the dataset
job_posting_cnt = df.count()
summary_df = pd.DataFrame({"Number of Job Postings": [job_posting_cnt]})
display(
    summary_df.style.hide(axis="index")
    .set_caption("Total Job Postings in Dataset")
)

# 2. Find the five most frequent job titles
if "TITLE_RAW" in df.columns:
    title_freq = (
        df.groupBy("TITLE_RAW")
          .agg(count("*").alias("posting_count"))
          .orderBy(desc("posting_count"))
          .limit(5)
          .toPandas()
    )

    title_freq = title_freq.rename(
        columns={
            "TITLE_RAW": "Job Title",
            "posting_count": "Postings"
        }
    )

    display(
        title_freq.style.hide(axis="index")
        .set_caption("Top 5 Most Common Job Titles")
    )


Number of Job Postings
72498


Job Title,Postings
Data Analyst,4201
Enterprise Architect,808
Senior Data Analyst,724
Business Intelligence Analyst,686
Data Modeler,281


In [5]:
from pyspark.sql.functions import avg, desc
import pandas as pd

# 3. Compute mean salary grouped by employment category
salary_field = None
employment_field = None

# Try to automatically detect the salary and employment-type columns
for col_name in df.columns:
    upper_name = col_name.upper()
    if "SALARY" in upper_name:
        salary_field = col_name
    if "EMPLOYMENT" in upper_name or "TYPE" in upper_name:
        employment_field = col_name

# If both columns are found, calculate the average salary per job type
if salary_field and employment_field:
    salary_by_type = (
        df.groupBy(employment_field)
          .agg(avg(salary_field).alias("Average Salary"))
          .orderBy(desc("Average Salary"))
          .toPandas()
          .rename(columns={employment_field: "Job Type"})
    )

    display(
        salary_by_type.style.hide(axis="index")
        .set_caption("Average Salary by Employment Type")
    )


Job Type,Average Salary
Remote,99576.870263
Hybrid Remote,93530.614447
[None],92765.666287
Not Remote,80340.732909
None,nan


In [6]:
from pyspark.sql.functions import count, desc
import pandas as pd

# 4. Identify the state that has the highest number of job postings
state_field = None

# Look for a column that appears to hold state information
for col_name in df.columns:
    if "STATE" in col_name.upper():
        state_field = col_name
        break

# If a state column is present, compute posting counts per state
if state_field:
    most_active_state = (
        df.groupBy(state_field)
          .agg(count("*").alias("Postings"))
          .orderBy(desc("Postings"))
          .limit(1)
          .toPandas()
          .rename(columns={state_field: "State"})
    )

    display(
        most_active_state.style.hide(axis="index")
        .set_caption("State with the Highest Number of Job Postings")
    )


State,Postings
48,8067
